Copyright (c) Microsoft Corporation. All rights reserved.

Licensed under the MIT License.

![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/MachineLearningNotebooks/how-to-use-azureml/work-with-data/datadrift-tutorial/datadrift-quickdemo.png)

In [1]:
# Check core SDK version number
import azureml.core

print('SDK version:', azureml.core.VERSION)

SDK version: 1.0.72


In [4]:
from azureml.core import Workspace

ws = Workspace.from_config()
ws

Workspace.create(name='SCUS-AzureML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG2')

In [5]:
# import Dataset class
from azureml.core import Dataset

# create target dataset 
#target = Dataset.Tabular.from_parquet_files(dstore.path('datadrift-data/**/data.parquet'))
# set the timestamp column
#target = target.with_timestamp_columns('datetime')
# register the target dataset
#target = target.register(ws, 'target')
# retrieve the dataset from the workspace by name
target = Dataset.get_by_name(ws, 'ufcfights')

In [6]:
target

{
  "source": [
    "('workspaceblobstore', 'UI/11-13-2019_075412_UTC/data.csv')"
  ],
  "definition": [
    "GetDatastoreFiles",
    "ParseDelimited",
    "DropColumns",
    "SetColumnTypes"
  ],
  "registration": {
    "id": "141372bf-b919-4620-998b-cf6ab0c4469d",
    "name": "ufcfights",
    "version": 1,
    "description": "UFC fights from https://www.kaggle.com/rajeevw/ufcdata \n1993-2019",
    "workspace": "Workspace.create(name='SCUS-AzureML', subscription_id='60582a10-b9fd-49f1-a546-c4194134bba8', resource_group='copetersRG2')"
  }
}

In [10]:
# import datetime 
from datetime import datetime

# set baseline dataset as January 2019 weather data
baseline = target.time_between(datetime(1993, 1, 1), datetime(1995, 1, 1))

In [12]:
from azureml.core.compute import AmlCompute, ComputeTarget

compute_name = 'cpu-cluster'

if compute_name in ws.compute_targets:
    compute_target = ws.compute_targets[compute_name]
    if compute_target and type(compute_target) is AmlCompute:
        print('found compute target. just use it. ' + compute_name)
else:
    print('creating a new compute target...')
    provisioning_config = AmlCompute.provisioning_configuration(vm_size='STANDARD_D4_V2', min_nodes=0, max_nodes=8)

    # create the cluster
    compute_target = ComputeTarget.create(ws, compute_name, provisioning_config)

    # can poll for a minimum number of nodes and for a specific timeout.
    # if no min node count is provided it will use the scale settings for the cluster
    compute_target.wait_for_completion(show_output=True, min_node_count=None, timeout_in_minutes=20)

    # For a more detailed view of current AmlCompute status, use get_status()
    print(compute_target.get_status().serialize())

found compute target. just use it. cpu-cluster


## Create data drift monitor

See [our documentation](http://aka.ms/datadrift) for a complete description for all of the parameters. 

In [13]:
from azureml.datadrift import DataDriftDetector, AlertConfiguration

alert_config = AlertConfiguration(['cody.peterson@microsoft.com']) # replace with your email to recieve alerts from the scheduled pipeline after enabling

monitor = DataDriftDetector.create_from_datasets(ws, 'ufcMonitor1', baseline, target, 
                                                      compute_target='cpu-cluster',    # compute target for scheduled pipeline and backfills 
                                                      frequency='Month',                     # how often to analyze target data
                                                      feature_list=None,                    # list of features to detect drift on
                                                      drift_threshold=1.0,                 # threshold from 0 to 1 for email alerting
                                                      latency=0,                            # SLA in hours for target data to arrive in the dataset
                                                      alert_config=alert_config)            # email addresses to send alert

2019-11-13 15:07:09,148 - azureml.datadrift._logging._telemetry_logger.datadriftdetector.create_from_datasets - ERROR - Operation returned an invalid status code 'DataDrift object does not exist' - activity_id:8ba0db8f-c0c6-4dc7-b50b-b4c1b5509603 activity_name:datadriftdetector.create_from_datasets activity_type:InternalCall workspace_name:SCUS-AzureML workspace_id:2cf9b7b7-0440-47de-b06e-0378f37e7162 baseline_dataset:None target_dataset:141372bf-b919-4620-998b-cf6ab0c4469d subscription_id:60582a10-b9fd-49f1-a546-c4194134bba8 workspace_location:southcentralus sdk_version:1.0.72 telemetry_component_name:azureml.datadrift


## Update data drift monitor

Many settings of the data drift monitor can be updated after creation. In this demo, we will update the `drift_threshold` and `feature_list`. See [our documentation](http://aka.ms/datadrift) for details on which settings can be changed.

In [10]:
#from azureml.datadrift import DataDriftDetector, AlertConfiguration

# get monitor by name
#monitor = DataDriftDetector.get_by_name(ws, 'weatherWeekly2018-2019')

In [11]:
# create feature list - need to exclude columns that naturally drift or increment over time, such as year, day, index
#columns  = list(baseline.take(1).to_pandas_dataframe())
#exclude  = ['year', 'day', 'version', '__index_level_0__']
#features = [col for col in columns if col not in exclude]

# update the feature list
#monitor  = monitor.update(feature_list=features)

In [14]:
from datetime import datetime

for year in range(1994, 2020):
    monitor.backfill(datetime(year, 1, 1), datetime(year+1, 1, 1))
    #monitor.backfill(datetime(year, 4, 1), datetime(year, 7, 1))
    #monitor.backfill(datetime(year, 7, 1), datetime(year, 10, 1))
    #monitor.backfill(datetime(year, 10, 1), datetime(year+1, 1, 1))

In [ ]:
# backfill for 1st half of the year
#backfill1 = monitor.backfill(datetime(2019, 1, 1), datetime(2019, 7, 1))
#backfill1

In [ ]:
# backfill for 2nd half of the year 
#backfill2 = monitor.backfill(datetime(2019, 9, 29), datetime(2019, 10, 23))
#backfill2

## Enable the monitor's pipeline schedule

Turn on a scheduled pipeline which will anlayze the target dataset for drift every `frequency`. Use the latency parameter to adjust the start time of the pipeline. For instance, if it takes 24 hours for my data processing pipelines for data to arrive in the target dataset, set latency to 24. 

In [ ]:
# enable the pipeline schedule and recieve email alerts
monitor.enable_schedule()

In [ ]:
# disable the pipeline schedule 
#monitor.disable_schedule()

## Query metrics and show results in Python

The below cell will plot some key data drift metrics, and can be used to query the results. Run `help(monitor.get_output)` for specifics on the object returned.

In [ ]:
# make sure the backfills have completed
backfill1.wait_for_completion()
backfill2.wait_for_completion()

In [ ]:
# get results from Python SDK (wait for backfills or monitor runs to finish)
results, metrics = monitor.get_output()

In [ ]:
# plot the results from Python SDK 
monitor.show()

## See results in Azure Machine Learning studio (Enterprise only)

The below cell will print a link to the monitor in the Azure Machine Learning studio, where the results can be viewed. Alertnatively, use the `show` or `get_results` to get and plot data drift results in Python.

In [ ]:
link = 'https://ml.azure.com/data/monitor/{}?wsid=/subscriptions/{}/resourcegroups/{}/workspaces/{}'.format(monitor.name, ws.subscription_id, ws.resource_group, ws.name)
print(link)

## Delete compute target

Do not delete the compute target if you intend to keep using it for the data drift monitor scheduled runs or otherwise. If the minimum nodes are set to 0, it will scale down soon after jobs are completed, and scale up the next time the cluster is needed.

In [ ]:
# optionally delete the compute target
#compute_target.delete()

## Next steps

  * See [our documentation](https://aka.ms/datadrift) or [Python SDK reference](https://docs.microsoft.com/python/api/overview/azure/ml/intro)
  * [Send requests or feedback](mailto:driftfeedback@microsoft.com) on data drift directly to the team
  * Please open issues with data drift here on GitHub or on StackOverflow if others are likely to run into the same issue